# Build Joint Dataset: Data Gathering, Join & Save

This notebook **fetches and joins** all sources into a single county–year dataset and **saves** it to CSV. Exploratory analysis is in **`joint_eda.ipynb`** (loads the saved file).

**Steps:**
1. Load USGS Pesticide (county–year)
2. Load Census ACS + NCHS urban–rural (county-level)
3. Load USDA Cropland + build county_crop_year_ag
4. IPM bridge: crop–geo scores and county–year collapse
5. Load CDC PLACES (2018 and 2019)
6. Join all on FIPS (+ YEAR for PLACES and county_year_collapsed)
7. **Save** → `data/joint_county_year_2018_2019.csv`

**Target years:** 2018 and 2019. **Join key:** 5-digit FIPS; PLACES and county_year_collapsed use FIPS + YEAR.

**Requirements:** `pandas`, `numpy`, `requests`. Optional: `rpy2` + R + CDCPLACES for PLACES.


In [5]:
import gc
import os
import re
import time
import warnings
from io import StringIO
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
import tempfile

# Optional: matplotlib, seaborn, geopandas (only needed for inline plots)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = sns = None
try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Optional: rpy2 for CDC PLACES (if missing, PLACES cell will use fallback)
try:
    from rpy2.robjects import r
    from rpy2.robjects.packages import importr
    from rpy2.robjects.vectors import StrVector
    HAS_RPY2 = True
except ImportError:
    HAS_RPY2 = False

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Stability / light-run config (set LIGHT_RUN=True if notebook crashes or runs out of memory) ---
LIGHT_RUN = False   # True: fewer CDL counties, lower concurrency, IPM skips PDF fetch
CDL_MAX_COUNTIES = 400 if LIGHT_RUN else None  # None = all pesticide counties
CDL_MAX_WORKERS = 3  # cap concurrency to avoid rate limits and memory spikes
IPM_SKIP_PDF_IN_LIGHT_RUN = True  # when LIGHT_RUN, use metadata-only IPM scores (no PDF/HTML fetch)


## 1. Load Pesticide Data (County–Year)

Load USGS pesticide estimates for 2016–2019. We keep **one row per county per year** (no aggregation across years) so the joint dataset supports temporal splits and time trends. We offer **multiple aggregation options** and **pesticide categories** by chemical class from Shekhar et al. (2024): [A systematic review of pesticide exposure, associated risks, and long-term human health impacts](https://pmc.ncbi.nlm.nih.gov/articles/PMC11664077/).

**Chemical classes** (linked to respiratory/neurological effects in the literature):
- **Organophosphates** (OP): cholinesterase inhibitors; respiratory, neurotoxic
- **Organochlorines** (OC): persistent; neurotoxic, endocrine disruption
- **Carbamates**: reversible AChE inhibitors; respiratory effects
- **Pyrethroids**: neurotoxic; respiratory distress
- **Triazines** (e.g. atrazine): herbicides; cancer associations
- **Chlorophenoxy** (e.g. 2,4-D): herbicides
- **Other/Unclassified**

In [6]:
# Pesticide URLs for 2016-2019
PEST_2016_17 = "https://www.sciencebase.gov/catalog/file/get/5e95c12282ce172707f2524e?f=__disk__62%2F83%2Fd3%2F6283d3501f1028b1ccc3976ea2e6de848bc2fef8&allowOpen=true"
PEST_2018 = "https://www.sciencebase.gov/catalog/file/get/6081a706d34e8564d686618e?f=__disk__58%2F6a%2Fed%2F586aed9a844eac0174a0600c8a7293ec4cda0265&allowOpen=true"
PEST_2019 = "https://www.sciencebase.gov/catalog/file/get/6081a924d34e8564d68661a1?f=__disk__08%2F42%2Fcd%2F0842cdac3a7d8b5056645a4dc08d1da96ad4e0b7&allowOpen=true"

df_pest = pd.read_csv(PEST_2016_17, sep="\t")
df_pest = df_pest[df_pest["YEAR"].isin([2016, 2017])]
df_pest = pd.concat([df_pest, pd.read_csv(PEST_2018, sep="\t"), pd.read_csv(PEST_2019, sep="\t")], ignore_index=True)

# Standardize FIPS
df_pest["STATE_FIPS"] = df_pest["STATE_FIPS_CODE"].astype(int).astype(str).str.zfill(2)
df_pest["COUNTY_FIPS"] = df_pest["COUNTY_FIPS_CODE"].astype(int).astype(str).str.zfill(3)
df_pest["FIPS"] = df_pest["STATE_FIPS"] + df_pest["COUNTY_FIPS"]
# USGS provides EPEST_LOW_KG and EPEST_HIGH_KG; use mean as single estimate (per datasheet)
df_pest["EPEST_MEAN_KG"] = df_pest[["EPEST_LOW_KG", "EPEST_HIGH_KG"]].mean(axis=1)

# --- Pesticide classification by chemical class (Shekhar et al., 2024, PMC11664077) ---
# Compounds matched by substring (case-insensitive). Order matters: more specific first.
CHEMICAL_CLASS_KEYWORDS = {
    "Organochlorine": ["ddt", "aldrin", "endosulfan", "endrin", "chlordane", "lindane", "dieldrin", "heptachlor", "toxaphene", "mirex", "methoxychlor"],
    "Organophosphate": ["chlorpyrifos", "malathion", "parathion", "diazinon", "dichlorvos", "fenthion", "phorate", "ethion", "acephate", "dimethoate", "methamidophos", "terbufos", "phosmet"],
    "Carbamate": ["carbaryl", "carbofuran", "propoxur", "bendiocarb", "aldicarb", "methomyl", "thiodicarb", "oxamyl", "pirimicarb"],
    "Pyrethroid": ["permethrin", "cypermethrin", "deltamethrin", "lambda-cyhalothrin", "cyhalothrin", "bifenthrin", "esfenvalerate", "fenvalerate", "tefluthrin", "pyrethroid"],
    "Triazine": ["atrazine", "propazine", "terbutryn", "cyanazine", "simazine", "terbuthylazine"],
    "Chlorophenoxy": ["2,4-d", "2,4-dichlorophenoxy", "mcpa", "mcpb", "dicamba", "2,4-db"],
    "Anilide": ["alachlor", "pretilachlor", "metolachlor", "acetochlor", "propanil"],
}

def assign_chemical_class(compound: str) -> str:
    """Assign chemical class from compound name (case-insensitive substring match)."""
    c = str(compound).lower()
    for cls, keywords in CHEMICAL_CLASS_KEYWORDS.items():
        if any(kw in c for kw in keywords):
            return cls
    return "Other"

df_pest["chemical_class"] = df_pest["COMPOUND"].apply(assign_chemical_class)

# --- Compute aggregations at county-year level (keep YEAR for temporal analysis) ---
# 1. Total: sum all compounds per county per year. Note that the calculation of total number of pesticides changes in 2019 to include fewer compounds.
pest_total = df_pest.groupby(["FIPS", "YEAR"])["EPEST_MEAN_KG"].sum().reset_index()
pest_total = pest_total.rename(columns={"EPEST_MEAN_KG": "pesticide_total_kg"})

# 2. By class: one column per chemical class (kg) per county-year
pest_by_class = df_pest.groupby(["FIPS", "YEAR", "chemical_class"])["EPEST_MEAN_KG"].sum().reset_index()
pest_class_wide = pest_by_class.pivot(index=["FIPS", "YEAR"], columns="chemical_class", values="EPEST_MEAN_KG").fillna(0).reset_index()
class_cols = [c for c in pest_class_wide.columns if c not in ("FIPS", "YEAR")]
pest_class_wide.columns = ["FIPS", "YEAR"] + [f"pesticide_{c.lower()}_kg" for c in class_cols]

# 3. Respiratory-relevant: sum only OP, Carbamate, Pyrethroid (strongest respiratory links) per county-year
resp_classes = ["Organophosphate", "Carbamate", "Pyrethroid"]
df_resp = df_pest[df_pest["chemical_class"].isin(resp_classes)]
pest_resp = df_resp.groupby(["FIPS", "YEAR"])["EPEST_MEAN_KG"].sum().reset_index()
pest_resp = pest_resp.rename(columns={"EPEST_MEAN_KG": "pesticide_respiratory_kg"})

# 4. By compound: one column per compound (kg) per county-year
def sanitize_compound_col(name):
    """Convert compound name to valid column name: pesticide_<sanitized>_kg"""
    s = re.sub(r"[^a-zA-Z0-9]", "_", str(name).strip()).lower()
    s = re.sub(r"_+", "_", s).strip("_")  # collapse multiple underscores
    return f"pesticide_{s}_kg" if s else "pesticide_unknown_kg"

pest_by_compound = df_pest.groupby(["FIPS", "YEAR", "COMPOUND"])["EPEST_MEAN_KG"].sum().reset_index()
pest_compound_wide = pest_by_compound.pivot(index=["FIPS", "YEAR"], columns="COMPOUND", values="EPEST_MEAN_KG").fillna(0).reset_index()
# Sanitize compound names for column headers; handle duplicates
seen = {}
new_cols = []
for c in pest_compound_wide.columns:
    if c in ("FIPS", "YEAR"):
        new_cols.append(c)
        continue
    name = sanitize_compound_col(c)
    seen[name] = seen.get(name, 0) + 1
    if seen[name] > 1:
        name = name.replace("_kg", f"_{seen[name]}_kg")
    new_cols.append(name)
pest_compound_wide.columns = new_cols

# Merge all into pest_county (one row per county-year)
pest_county = pest_total.merge(pest_class_wide, on=["FIPS", "YEAR"], how="left").merge(pest_resp, on=["FIPS", "YEAR"], how="left")
pest_county = pest_county.merge(pest_compound_wide, on=["FIPS", "YEAR"], how="left")
pest_county["pesticide_respiratory_kg"] = pest_county["pesticide_respiratory_kg"].fillna(0)

print("Pesticide (county-year):", pest_county.shape)
print("  - Total + respiratory +", pest_class_wide.shape[1] - 2, "class cols +", pest_compound_wide.shape[1] - 2, "compound cols (one row per county-year)")
print("Chemical class distribution:", df_pest["chemical_class"].value_counts())
pest_county.head()


# ultimately it can be ok to to have a large number of features from the chemical compounds and we can observe if there are features downwighted seariously by the model itself. 

Pesticide (county-year): (12259, 447)
  - Total + respiratory + 8 class cols + 435 compound cols (one row per county-year)
Chemical class distribution: chemical_class
Other              983889
Pyrethroid          62367
Anilide             47775
Organophosphate     42213
Chlorophenoxy       36944
Triazine            21907
Carbamate           17467
Organochlorine          7
Name: count, dtype: int64


,FIPS,YEAR,pesticide_total_kg,pesticide_anilide_kg,pesticide_carbamate_kg,pesticide_chlorophenoxy_kg,pesticide_organochlorine_kg,pesticide_organophosphate_kg,pesticide_other_kg,pesticide_pyrethroid_kg,...,pesticide_trifluralin_kg,pesticide_triflusulfuron_kg,pesticide_trinexapac_kg,pesticide_triticonazole_kg,pesticide_uniconazole_kg,pesticide_vinclozolin_kg,pesticide_zeta_cypermethrin_kg,pesticide_zinc_kg,pesticide_ziram_kg,pesticide_zoxamide_kg
0,01001,2016,36955.10,2882.95,166.95,1642.40,0.0,241.90,30739.35,192.45,...,68.5,0.0,0.0,0.0,0.0,0.0,3.25,0.0,0.0,0.0
1,01001,2017,29132.90,3654.30,322.55,1535.95,0.0,362.85,22591.60,314.65,...,703.2,0.0,0.0,0.0,0.0,0.0,12.30,0.0,0.0,0.0
2,01001,2018,34773.85,3690.60,397.30,6995.35,0.0,2033.75,20854.80,119.65,...,192.2,0.0,0.0,0.0,0.0,0.0,8.50,0.0,0.0,0.0
3,01001,2019,21120.00,2582.55,9.70,5230.55,0.0,763.95,11639.25,16.80,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
4,01003,2016,324752.70,70808.75,460.80,8460.75,0.0,9263.35,213589.00,1580.65,...,109.6,0.0,0.0,0.0,0.0,0.0,36.70,0.0,0.0,0.0


## 2. Load Census Demographics (County-level)

ACS 5-year (2019) for population, median age, income; **race percentages** (pct_white, pct_black, pct_asian, pct_hispanic). **Rural/urban:** NCHS 6-level classification (1–6) from [CDC NCHS Urban-Rural](https://www.cdc.gov/nchs/data-analysis-tools/urban-rural.html), merged on county FIPS (state FIPS + county FIPS).

In [7]:
CENSUS_BASE = "https://api.census.gov/data"
VARS = ["NAME", "B01003_001E", "B01002_001E", "B19013_001E", "B03002_001E", "B03002_003E", "B03002_004E", "B03002_006E", "B03002_012E"]

resp = requests.get(f"{CENSUS_BASE}/2019/acs/acs5", params={"get": ",".join(VARS), "for": "county:*", "in": "state:*"})
resp.raise_for_status()
data = resp.json()
census = pd.DataFrame(data[1:], columns=data[0])

census["FIPS"] = census["state"].astype(str).str.zfill(2) + census["county"].astype(str).str.zfill(3)
for c in ["B01003_001E", "B01002_001E", "B19013_001E", "B03002_001E", "B03002_003E", "B03002_004E", "B03002_006E", "B03002_012E"]:
    census[c] = pd.to_numeric(census[c], errors="coerce")

census = census.rename(columns={"B01003_001E": "population", "B01002_001E": "median_age", "B19013_001E": "median_income", "B03002_001E": "pop_race"})
# Race percentages (demographic confounders for respiratory health)
for race, col in [("pct_white", "B03002_003E"), ("pct_black", "B03002_004E"), ("pct_asian", "B03002_006E"), ("pct_hispanic", "B03002_012E")]:
    census[race] = 100 * census[col] / census["pop_race"].replace(0, np.nan)

# Rural/urban: NCHS 6-level classification (1–6), county-level; merge on FIPS = state FIPS (2) + county FIPS (3)
# https://www.cdc.gov/nchs/data-analysis-tools/urban-rural.html  (2023 scheme: CODE2023)
NCHS_URBAN_RURAL_URL = "https://www.cdc.gov/nchs/data/data-analysis/NCHSurb-rural-codes.csv"
try:
    r_nchs = requests.get(NCHS_URBAN_RURAL_URL, timeout=30)
    r_nchs.raise_for_status()
    nchs = pd.read_csv(StringIO(r_nchs.text), dtype=str)
    nchs.columns = [c.strip() for c in nchs.columns]
    # Per CDC file docs: STFIPS, CTYFIPS; full FIPS = zfill(2) + zfill(3)
    state_col = "STFIPS" if "STFIPS" in nchs.columns else next((c for c in nchs.columns if "state" in c.lower() and "fips" in c.lower()), nchs.columns[0])
    county_col = "CTYFIPS" if "CTYFIPS" in nchs.columns else next((c for c in nchs.columns if "county" in c.lower() and "fips" in c.lower()), nchs.columns[1])
    nchs["FIPS"] = nchs[state_col].astype(str).str.zfill(2) + nchs[county_col].astype(str).str.zfill(3)
    code_col = "CODE2023" if "CODE2023" in nchs.columns else next((c for c in nchs.columns if "2023" in c or "code" in c.lower()), None)
    if code_col is None:
        code_col = [c for c in nchs.columns if c not in (state_col, county_col, "FIPS", "CTYNAME", "ST_ABBREV")][0]
    nchs["nchs_urban_rural"] = pd.to_numeric(nchs[code_col], errors="coerce")
    nchs = nchs[["FIPS", "nchs_urban_rural"]].dropna(subset=["nchs_urban_rural"])
    census = census.merge(nchs, on="FIPS", how="left")
    print("Rural/urban: added nchs_urban_rural (1–6) from CDC NCHS; 1=Large central metro, 2=Large fringe, 3=Medium, 4=Small metro, 5=Micropolitan, 6=Noncore.")
except Exception as e:
    print("NCHS urban-rural failed:", e)
    nchs_path = os.path.join(os.path.dirname(os.getcwd()), "data", "NCHSurb-rural-codes.csv")
    if os.path.isfile(nchs_path):
        nchs = pd.read_csv(nchs_path, dtype=str)
        nchs.columns = [c.strip() for c in nchs.columns]
        state_col = "STFIPS" if "STFIPS" in nchs.columns else [c for c in nchs.columns if "state" in c.lower() and "fips" in c.lower()][0]
        county_col = "CTYFIPS" if "CTYFIPS" in nchs.columns else [c for c in nchs.columns if "county" in c.lower() and "fips" in c.lower()][0]
        code_col = "CODE2023" if "CODE2023" in nchs.columns else [c for c in nchs.columns if "code" in c.lower()][0]
        nchs["FIPS"] = nchs[state_col].astype(str).str.zfill(2) + nchs[county_col].astype(str).str.zfill(3)
        nchs["nchs_urban_rural"] = pd.to_numeric(nchs[code_col], errors="coerce")
        census = census.merge(nchs[["FIPS", "nchs_urban_rural"]].dropna(subset=["nchs_urban_rural"]), on="FIPS", how="left")
        print("Loaded NCHS urban-rural from local data/NCHSurb-rural-codes.csv")
    else:
        census["nchs_urban_rural"] = np.nan

print("Census (county-level):", census.shape)
census.head()

Rural/urban: added nchs_urban_rural (1–6) from CDC NCHS; 1=Large central metro, 2=Large fringe, 3=Medium, 4=Small metro, 5=Micropolitan, 6=Noncore.
Census (county-level): (3220, 17)


,NAME,population,median_age,median_income,pop_race,B03002_003E,B03002_004E,B03002_006E,B03002_012E,state,county,FIPS,pct_white,pct_black,pct_asian,pct_hispanic,nchs_urban_rural
0,"Fayette County, Illinois",21565,41.9,46650,21565,19868,1007,116,403,17,051,17051,92.130767,4.669604,0.537909,1.868769,6.0
1,"Logan County, Illinois",29003,40.1,57308,29003,25049,1984,218,985,17,107,17107,86.366928,6.840672,0.751646,3.396200,5.0
2,"Saline County, Illinois",23994,42.2,44090,23994,21957,632,173,428,17,165,17165,91.510378,2.633992,0.721014,1.783779,6.0
3,"Lake County, Illinois",701473,38.4,89427,701473,432361,45923,53654,152141,17,097,17097,61.636157,6.546653,7.648762,21.688789,2.0
4,"Massac County, Illinois",14219,43.5,47481,14219,12547,830,29,419,17,127,17127,88.241086,5.837260,0.203952,2.946761,4.0


## 3. Load Cropland Data (County-level)

The CropScape API returns **acreage by land-cover category** (e.g., Corn, Soybeans, Cotton, Forest, Developed). We extract multiple factors beyond total acres.

**Fetch strategy:** We fetch CDL for **all** pesticide counties using **parallel requests**. A short **probe** (15 test requests) runs first to find a safe concurrency level — the API doesn't publish rate limits, so we try 10 workers, then 5, 2, or 1 if we hit 429/503 errors.

**Cropland factors extracted** (likely relevant for pesticide/health modeling):

| Factor | Rationale |
|--------|------------|
| `cropland_acres` | Total cultivated cropland — baseline agricultural intensity |
| `pct_cropland` | Share of county in crops — intensity relative to land area |
| `corn_acres`, `soybean_acres`, `cotton_acres` | Major row crops — different pesticide profiles (e.g., atrazine on corn) |
| `fruit_veg_acres` | Orchards, vineyards, vegetables — often higher pesticide use per acre |
| `hay_acres` | Alfalfa/hay — moderate use |
| `cropland_diversity` | Number of crop types — monoculture vs. mixed farming |

In [8]:
CDL_API = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

def get_county_cdl(year: int, fips: str) -> pd.DataFrame:
    params = {"year": year, "fips": fips, "format": "csv"}
    r = requests.get(CDL_API, params=params)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    ns = {"ns": "http://cropscape.csiss.gmu.edu/CDLService/"}
    url_elem = root.find(".//ns:returnURL", ns) or root.find(".//returnURL")
    data_url = url_elem.text.strip() if url_elem is not None else f"https://nassgeodata.gmu.edu/webservice/nass_data_cache/byfips/CDL_{year}_{fips}.csv"
    r2 = requests.get(data_url)
    r2.raise_for_status()
    temp_df = pd.read_csv(StringIO(r2.text))
    temp_df.columns = temp_df.columns.str.strip() # fixing whitespace
    temp_df['Category'] = temp_df['Category'].str.strip() # fixing whitespace
    return temp_df

# Fetch cropland for pesticide counties (parallel requests; use subset if LIGHT_RUN)
all_fips = pest_county["FIPS"].astype(str).str.zfill(5).unique().tolist()
if CDL_MAX_COUNTIES is not None:
    all_fips = all_fips[: int(CDL_MAX_COUNTIES)]
    print("CDL: using first", len(all_fips), "counties (LIGHT_RUN or CDL_MAX_COUNTIES).")

# Probe: discover safe concurrency (API doesn't publish rate limits)
# Try a small batch; if we get 429/503, reduce workers and retry
def probe_safe_workers(max_workers_start=10, probe_size=15):
    probe_fips = all_fips[:probe_size]
    for n in [max_workers_start, 5, 2, 1]:
        errors = 0
        with ThreadPoolExecutor(max_workers=n) as ex:
            futures = [ex.submit(get_county_cdl, 2019, f) for f in probe_fips]
            for f in as_completed(futures):
                try:
                    f.result()
                except requests.exceptions.HTTPError as e:
                    if e.response.status_code in (429, 503):
                        errors += 1
                except Exception:
                    errors += 1
        if errors == 0:
            print(f"Probe OK with {n} worker(s). Using N_WORKERS={n}.")
            return n
        print(f"Probe with {n} worker(s): {errors} rate-limit/server errors. Trying fewer...")
        time.sleep(2)
    return 1

N_WORKERS = probe_safe_workers(max_workers_start=10, probe_size=15)

CROP_GROUPS = {
    "corn_acres": ["corn", "sweet corn", "maize"],
    "soybean_acres": ["soybean", "soybeans", "soy"],
    "cotton_acres": ["cotton"],
    "wheat_acres": ["wheat", "durum", "spring wheat", "winter wheat"],
    "hay_acres": ["hay", "alfalfa"],
    "fruit_veg_acres": [
        "orchard", "orchards", "vineyard", "vineyards", "fruit", "fruits", "vegetable", "vegetables",
        "grape", "grapes", "apple", "apples", "citrus", "berry", "berries", "tomato", "tomatoes",
        "peach", "peaches", "almond", "almonds", "potato", "potatoes", "melon", "melons", "lettuce",
        "onion", "onions", "carrot", "carrots", "broccoli", "pepper", "peppers", "strawberry", "strawberries"
    ],
    "rice_acres": ["rice"],
    "sorghum_acres": ["sorghum"],
}
NON_CROP = ["developed", "open water", "water", "forest", "wetland", "barren", "shrubland", "grassland", "pasture"]

CANONICAL_CROP_KEYWORDS = {
    "corn": ["sweet corn", "maize", "corn"],
    "soybean": ["soybeans", "soybean", "soy"],
    "cotton": ["cotton"],
    "wheat": ["durum", "spring wheat", "winter wheat", "wheat"],
    "hay": ["alfalfa", "hay"],
    "fruit_veg": [
        "strawberries", "strawberry", "vegetables", "vegetable", "vineyards", "vineyard", "orchards", "orchard",
        "potatoes", "potato", "tomatoes", "tomato", "berries", "berry", "apples", "apple", "almonds", "almond",
        "peaches", "peach", "grapes", "grape", "citrus", "melons", "melon", "lettuce", "onions", "onion",
        "carrots", "carrot", "broccoli", "peppers", "pepper", "fruit", "fruits"
    ],
    "rice": ["rice"],
    "sorghum": ["sorghum"],
}
CROP_FAMILY_MAP = {
    "corn": "field_crop",
    "soybean": "field_crop",
    "cotton": "field_crop",
    "wheat": "field_crop",
    "rice": "field_crop",
    "sorghum": "field_crop",
    "hay": "forage",
    "fruit_veg": "specialty_crop",
    "other_crop": "other_crop",
}

_SORTED_CROP_KEYWORDS = sorted(
    [(keyword, crop) for crop, keywords in CANONICAL_CROP_KEYWORDS.items() for keyword in keywords],
    key=lambda kv: len(kv[0]),
    reverse=True,
)

def normalize_crop_label(name: str) -> str:
    text = re.sub(r"[^a-z0-9]+", " ", str(name or "").lower()).strip()
    for keyword, crop in _SORTED_CROP_KEYWORDS:
        if re.search(rf"\b{re.escape(keyword)}\b", text):
            return crop
    return "other_crop"

def crop_family(crop: str) -> str:
    return CROP_FAMILY_MAP.get(str(crop), "other_crop")

def cdl_to_long(df_c: pd.DataFrame, fips: str, year: int) -> list:
    """Convert CDL stat response to long format (FIPS, year, crop, crop_family, acres) for county_crop_year_ag."""
    acre_col = next((c for c in ["Acres", "ACRES", "acres"] if c in df_c.columns), None)
    if acre_col is None:
        cand = [c for c in df_c.columns if "acre" in c.lower() or "area" in c.lower()]
        acre_col = cand[0] if cand else df_c.select_dtypes(include=[np.number]).columns[0]
    name_col = next((c for c in ["Category", "Crop", "Class_Name", "NAME"] if c in df_c.columns), df_c.columns[0])
    df_c = df_c.copy()
    df_c["_name"] = df_c[name_col].astype(str).str.lower()
    non_crop_mask = df_c["_name"].str.contains("|".join(NON_CROP), case=False, na=False)
    df_c = df_c.loc[~non_crop_mask]
    rows = []
    for _, r in df_c.iterrows():
        name, ac = r["_name"], r[acre_col]
        if ac <= 0:
            continue
        crop = normalize_crop_label(name)
        rows.append({"FIPS": fips, "year": year, "crop": crop, "crop_family": crop_family(crop), "acres": ac})
    return rows

def extract_crop_features(df_c: pd.DataFrame) -> dict:
    """Extract cropland features from CDL stat response."""
    acre_col = next((c for c in ["Acres", "ACRES", "acres"] if c in df_c.columns), None)
    if acre_col is None:
        cand = [c for c in df_c.columns if "acre" in c.lower() or "area" in c.lower()]
        acre_col = cand[0] if cand else df_c.select_dtypes(include=[np.number]).columns[0]
    name_col = next((c for c in ["Category", "Crop", "Class_Name", "NAME"] if c in df_c.columns), df_c.columns[0])
    df_c = df_c.copy()
    df_c["_name"] = df_c[name_col].astype(str).str.lower()
    total_area = df_c[acre_col].sum()
    crop_mask = ~df_c["_name"].str.contains("|".join(NON_CROP), case=False, na=False)
    cropland_total = df_c.loc[crop_mask, acre_col].sum()
    out = {"cropland_acres": cropland_total, "total_area_acres": total_area, "pct_cropland": 100 * cropland_total / total_area if total_area > 0 else 0}
    for feat, keywords in CROP_GROUPS.items():
        mask = df_c["_name"].str.contains("|".join(re.escape(k) for k in keywords), case=False, na=False)
        out[feat] = df_c.loc[mask, acre_col].sum()
    out["cropland_diversity"] = crop_mask.sum()
    return out

CDL_YEAR = 2019

def fetch_one(fips):
    try:
        df_c = get_county_cdl(CDL_YEAR, fips)
        feats = extract_crop_features(df_c)
        feats["FIPS"] = fips
        long_rows = cdl_to_long(df_c, fips, CDL_YEAR)
        return (feats, long_rows)
    except Exception:
        return None

cropland_rows = []
county_crop_year_ag_rows = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(fetch_one, f): f for f in all_fips}
    for i, fut in enumerate(as_completed(futures), 1):
        out = fut.result()
        if out is not None:
            feats, long_rows = out
            cropland_rows.append(feats)
            county_crop_year_ag_rows.extend(long_rows)
        if i % 250 == 0:
            print(f"Fetched {i}/{len(all_fips)} counties")

cropland = pd.DataFrame(cropland_rows)
print("Cropland feature rows:", cropland.shape)
print("Columns:", list(cropland.columns))

# Build county_crop_year_ag: long table (FIPS, year, crop, crop_family, acres, share_county_crop_acres)
# Repeat CDL_YEAR composition for all analysis years 2016-2019 so joint join has one row per county-year
analysis_years = sorted(pest_county["YEAR"].dropna().unique().astype(int).tolist())
ccy_ag = pd.DataFrame(county_crop_year_ag_rows)
if len(ccy_ag) > 0:
    ccy_list = []
    for yr in analysis_years:
        ccy_yr = ccy_ag.copy()
        ccy_yr["year"] = yr
        ccy_list.append(ccy_yr)
    ccy_ag = pd.concat(ccy_list, ignore_index=True)
    tot_acres = ccy_ag.groupby(["FIPS", "year"])["acres"].transform("sum")
    ccy_ag["share_county_crop_acres"] = np.where(tot_acres > 0, ccy_ag["acres"] / tot_acres, np.nan)
    ccy_ag["crop_value"] = np.nan
    ccy_ag["share_county_crop_value"] = np.nan
    county_crop_year_ag = ccy_ag
    print("County-crop-year (county_crop_year_ag):", county_crop_year_ag.shape)
    if "crop" in county_crop_year_ag.columns:
        print("  Crop mix:", county_crop_year_ag["crop"].value_counts().head(12).to_dict())
else:
    county_crop_year_ag = pd.DataFrame(columns=["FIPS", "year", "crop", "crop_family", "acres", "share_county_crop_acres", "crop_value", "share_county_crop_value"])

cropland.head()


Probe OK with 10 worker(s). Using N_WORKERS=10.
Fetched 250/3066 counties
Fetched 500/3066 counties
Fetched 750/3066 counties
Fetched 1000/3066 counties
Fetched 1250/3066 counties
Fetched 1500/3066 counties
Fetched 1750/3066 counties
Fetched 2000/3066 counties
Fetched 2250/3066 counties
Fetched 2500/3066 counties
Fetched 2750/3066 counties
Fetched 3000/3066 counties
Cropland feature rows: (3037, 13)
Columns: ['cropland_acres', 'total_area_acres', 'pct_cropland', 'corn_acres', 'soybean_acres', 'cotton_acres', 'wheat_acres', 'hay_acres', 'fruit_veg_acres', 'rice_acres', 'sorghum_acres', 'cropland_diversity', 'FIPS']
County-crop-year (county_crop_year_ag): (286828, 8)
  Crop mix: {'other_crop': 144452, 'corn': 30888, 'fruit_veg': 27396, 'hay': 22700, 'soybean': 22144, 'wheat': 16008, 'sorghum': 14712, 'cotton': 6904, 'rice': 1624}


,cropland_acres,total_area_acres,pct_cropland,corn_acres,soybean_acres,cotton_acres,wheat_acres,hay_acres,fruit_veg_acres,rice_acres,sorghum_acres,cropland_diversity,FIPS
0,13745.0,385952.2,3.561322,667.1,18.3,661.4,144.3,12082.0,0.2,0.0,4.3,23,01017
1,27162.3,391897.7,6.930967,2604.7,2631.8,5133.3,0.0,16594.2,0.2,0.0,17.1,18,01015
2,9472.6,400772.7,2.363584,711.4,288.9,336.3,3.8,7087.7,0.6,0.0,0.7,21,01007
3,24466.2,497815.3,4.914714,2245.7,51.5,2398.1,9.8,17892.1,0.2,0.0,147.2,21,01013
4,51446.3,386553.2,13.308983,2481.5,394.7,11046.3,42.7,35341.6,27.1,0.0,220.8,25,01001


In [9]:
cropland.describe()

,cropland_acres,total_area_acres,pct_cropland,corn_acres,soybean_acres,cotton_acres,wheat_acres,hay_acres,fruit_veg_acres,rice_acres,sorghum_acres,cropland_diversity
count,3.037000e+03,3.037000e+03,3037.000000,3037.000000,3037.000000,3037.000000,3037.000000,3037.000000,3037.000000,3037.000000,3037.000000,3037.000000
mean,1.164712e+05,6.281408e+05,24.651953,30033.663912,25815.666711,4968.838492,14691.683932,17429.373527,1795.048765,845.361376,2117.070069,23.611129
std,1.421101e+05,8.463548e+05,25.074805,49899.607738,44471.578549,23697.522926,48016.708202,26033.734829,17252.316610,6978.967041,9233.165402,8.735231
min,1.080000e+01,1.584630e+04,0.000397,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
25%,1.552000e+04,2.838791e+05,3.594766,489.500000,15.600000,0.000000,11.600000,2787.900000,0.200000,0.000000,1.300000,18.000000
50%,6.233390e+04,4.049092e+05,14.691719,5736.000000,2150.100000,0.000000,192.200000,8358.100000,4.200000,0.000000,19.100000,22.000000
75%,1.754877e+05,5.998976e+05,41.695406,37297.200000,34141.600000,78.500000,2426.100000,20844.300000,75.200000,0.000000,212.400000,28.000000
max,1.300430e+06,1.286771e+07,91.601879,322829.900000,436053.900000,435876.300000,679236.500000,336196.200000,597483.000000,137967.600000,130503.200000,68.000000


## 3b. IPM bridge: crop–geo scores and county–year collapse

IPM sources (Crop Profiles, PMSPs) are **crop × state/region × document**, not county-level. This section implements **section-aware scoring** inline (lexicons for monitoring, nonchemical, chemical, decision support, dependency, resistance management; weak metadata priors; optional PDF fallback). We integrate by:

1. **county_crop_year_ag** — crop composition (acres, shares) per county-year (built above).
2. **crop_geo_doc_ipm** — IPM document-derived scores per crop × state × document year (National IPM Database API; section-aware subscales + ipm_breadth_index, chemical_reliance_index).
3. **county_crop_year_ipm** — aggregate to one row per (crop, state_fips), then join county crops to crop–state IPM on crop + state.
4. **county_year_collapsed** — roll up to county-year with **acreage-weighted** (primary) and **value-weighted** (sensitivity) IPM breadth and chemical reliance.

Interpretation: *"The IPM characteristics of the crop mix grown in this county"* — not county-level adoption.

In [10]:
# --- crop_geo_doc_ipm: section-aware scores (inline; National IPM Database API) ---
# Expected runtime: ~5-15 min with PDF/HTML fallback (API + per-doc fetches). Coverage-first mode keeps all docs,
# then lets aggregation / match tiers decide how to back off when exact crop-state matches are sparse.
import re
import requests
import numpy as np
import pandas as pd

IPM_API_BASE = "https://ipmdata.ipmcenters.org/rest/ipmdata_ipmcenters_org_restapi"
ANALYSIS_YEARS_IPM = [2018, 2019]
USE_MOST_RECENT_DOC_ONLY = False
USE_STRUCTURED_TEXT = True
USE_PDF_FALLBACK = True
PDF_TIMEOUT = 20
REQUEST_TIMEOUT = 30

if "CROP_FAMILY_MAP" not in globals():
    CROP_FAMILY_MAP = {
        "corn": "field_crop", "soybean": "field_crop", "cotton": "field_crop", "wheat": "field_crop",
        "rice": "field_crop", "sorghum": "field_crop", "hay": "forage", "fruit_veg": "specialty_crop",
        "other_crop": "other_crop",
    }

TITLE_TO_CROP = {
    "sweet corn": "corn", "maize": "corn", "corn": "corn",
    "soybeans": "soybean", "soybean": "soybean", "soy": "soybean",
    "cotton": "cotton",
    "durum": "wheat", "spring wheat": "wheat", "winter wheat": "wheat", "wheat": "wheat",
    "alfalfa": "hay", "hay": "hay", "forage": "hay",
    "rice": "rice",
    "grain sorghum": "sorghum", "sorghum": "sorghum",
    "strawberry": "fruit_veg", "strawberries": "fruit_veg", "vegetable": "fruit_veg", "vegetables": "fruit_veg",
    "vineyard": "fruit_veg", "vineyards": "fruit_veg", "orchard": "fruit_veg", "orchards": "fruit_veg",
    "potato": "fruit_veg", "potatoes": "fruit_veg", "tomato": "fruit_veg", "tomatoes": "fruit_veg",
    "berry": "fruit_veg", "berries": "fruit_veg", "apple": "fruit_veg", "apples": "fruit_veg",
    "almond": "fruit_veg", "almonds": "fruit_veg", "peach": "fruit_veg", "peaches": "fruit_veg",
    "grape": "fruit_veg", "grapes": "fruit_veg", "citrus": "fruit_veg", "melon": "fruit_veg", "melons": "fruit_veg",
    "lettuce": "fruit_veg", "onion": "fruit_veg", "onions": "fruit_veg", "carrot": "fruit_veg", "carrots": "fruit_veg",
    "broccoli": "fruit_veg", "pepper": "fruit_veg", "peppers": "fruit_veg", "fruit": "fruit_veg", "fruits": "fruit_veg",
    "bean": "other_crop", "beans": "other_crop", "dry bean": "other_crop", "peanut": "other_crop",
    "sunflower": "other_crop", "canola": "other_crop", "barley": "other_crop", "oat": "other_crop",
}
_SORTED_TITLE_CROPS = sorted(TITLE_TO_CROP.items(), key=lambda kv: len(kv[0]), reverse=True)

# Lexicons (ambiguous terms like organic, application, treatment, timing, pollinator, integrated, reduced-risk removed to reduce noise)
LEXICONS = {
    "monitoring": [r"scout(?:ing)?", r"monitor(?:ing)?", r"trap(?:ping|s)?", r"sampl(?:e|ing)", r"thresholds?", r"economic threshold", r"degree[- ]day", r"forecast(?:ing)?", r"surveillance", r"detection", r"sweep(?:ing)?", r"action threshold", r"monitoring (?:and )?scouting"],
    "nonchemical": [r"cultural control", r"biological control", r"mechanical control", r"physical control", r"crop rotation", r"sanitation", r"resistant variet(?:y|ies)", r"pruning", r"tillage", r"mulch(?:ing)?", r"habitat management", r"natural enemies", r"pheromone disruption", r"nonchemical", r"non-chemical", r"biologicals?", r"beneficial (?:insects?|organisms?)", r"cultural practices", r"cover crop", r"intercropping"],
    "chemical": [r"pesticides?", r"insecticides?", r"herbicides?", r"fungicides?", r"chemical control", r"spray(?:ing| schedule)?", r"active ingredient", r"pre-?emergence", r"post-?emergence", r"chemigation", r"foliar", r"soil applied", r"registered (?:pesticide|chemical)", r"chemical management", r"conventional (?:pesticide|chemical)"],
    "decision_support": [r"thresholds?", r"action threshold", r"economic threshold", r"targeted", r"integrated pest", r"integrated management", r"beneficials?", r"application timing", r"ipm (?:practices?|approach)", r"decision[- ]?making", r"when to (?:spray|treat|apply)", r"scouting (?:for|to)"],
    "dependency": [r"limited alternatives?", r"few effective options?", r"critical use", r"loss of registration", r"resistance concern", r"lack of nonchemical options", r"dependency", r"reliance", r"primary (?:control|tool)", r"conventional (?:control|management)"],
    "resistance_management": [r"resistance management", r"mode of action", r"rotat(?:e|ion) of chemistr(?:y|ies)", r"anti-resistance", r"insecticide resistance", r"herbicide resistance", r"moa", r"tank[- ]?mix", r"rotate (?:modes|chemistry)"],
}

SECTION_PATTERNS = {
    "production_practices": [r"production practices", r"production information", r"production facts", r"growing practices"],
    "monitoring": [r"monitoring", r"scouting", r"sampling", r"threshold", r"pest identification", r"detection"],
    "cultural_controls": [r"cultural control", r"cultural practices", r"cultural management", r"nonchemical"],
    "biological_controls": [r"biological control", r"natural enemies", r"biologicals", r"biocontrol"],
    "physical_controls": [r"physical control", r"mechanical control", r"physical and mechanical"],
    "chemical_controls": [r"chemical control", r"pesticides", r"herbicides", r"insecticides", r"fungicides", r"chemical management", r"pest management (?:strategies|practices)", r"weed management", r"insect management", r"disease management"],
    "research_priorities": [r"research priorities", r"regulatory priorities", r"education priorities", r"priorities", r"transition priorities", r"worker (?:activities|protection)", r"pollinator"],
}

def _compile(patterns):
    return [re.compile(p, re.I) for p in patterns]

COMPILED_LEXICONS = {k: _compile(v) for k, v in LEXICONS.items()}
COMPILED_SECTION_PATTERNS = {k: _compile(v) for k, v in SECTION_PATTERNS.items()}

def _ipm_get(path, params=None):
    r = requests.get(f"{IPM_API_BASE}{path}", params=params or {}, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()

def _ipm_response_to_rows(obj):
    if isinstance(obj, dict) and "COLUMNS" in obj and "DATA" in obj:
        cols = [str(c).strip() for c in obj["COLUMNS"]]
        return [dict(zip(cols, row)) for row in obj["DATA"]]
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        return [obj]
    return []

def _get(d, *keys):
    for k in keys:
        v = d.get(k) or d.get(k.upper() if hasattr(k, "upper") else k) or d.get(k.lower() if hasattr(k, "lower") else k)
        if v is not None:
            return v
    return None

def _parse_doc_year(sourcedate):
    if pd.isna(sourcedate):
        return np.nan
    s = str(sourcedate).strip()
    m = re.search(r"(19|20)\d{2}", s)
    return int(m.group(0)) if m else np.nan

def _crop_family(crop):
    return CROP_FAMILY_MAP.get(str(crop), "other_crop")

def _crop_from_title(title):
    text = re.sub(r"[^a-z0-9]+", " ", str(title or "").lower()).strip()
    for keyword, crop in _SORTED_TITLE_CROPS:
        if re.search(rf"\b{re.escape(keyword)}\b", text):
            return crop
    return "other_crop"

def _bounded(x, lo=0.0, hi=1.0):
    return min(hi, max(lo, x))

def _normalize_rate(count, denom):
    return count / denom if denom > 0 else 0.0

def _metadata_priors(doc_type, doc_year, crop):
    monitoring = nonchemical = chemical = decision_support = dependency = resistance_management = 0.0
    doc_type_str = str(doc_type or "").lower()
    if "pmsp" in doc_type_str:
        decision_support += 0.05
        dependency += 0.05
    if pd.notna(doc_year):
        if doc_year >= 2010:
            monitoring += 0.03
            nonchemical += 0.03
            resistance_management += 0.03
        elif doc_year < 2000:
            chemical += 0.04
            dependency += 0.04
    if crop == "fruit_veg":
        monitoring += 0.03
        nonchemical += 0.03
    return {"monitoring_prior": _bounded(monitoring), "nonchemical_prior": _bounded(nonchemical), "chemical_prior": _bounded(chemical), "decision_support_prior": _bounded(decision_support), "dependency_prior": _bounded(dependency), "resistance_management_prior": _bounded(resistance_management)}

def _fetch_structured_report_text(source_id):
    return None

def _fetch_pdf_text(url):
    if not url:
        return None
    try:
        from io import BytesIO
        r = requests.get(str(url), timeout=PDF_TIMEOUT)
        r.raise_for_status()
        raw = r.content
        if len(raw) < 500:
            return None
        text = None
        try:
            from pdfminer.high_level import extract_text as pdf_extract_text
            text = pdf_extract_text(BytesIO(raw))
        except Exception:
            text = None
        if not text or len(text.strip()) < 150:
            try:
                import fitz
                doc = fitz.open(stream=raw, filetype="pdf")
                parts = [doc.load_page(i).get_text() for i in range(len(doc))]
                doc.close()
                text = "\n".join(parts) if parts else ""
            except Exception:
                pass
        if not text or len(text.strip()) < 150:
            return None
        return text.strip()
    except Exception:
        return None

def _split_sections_from_text(text):
    if not text:
        return {}
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    joined = "\n".join(lines)
    lower = joined.lower()
    sections = {}
    heading_candidates = ["production practices", "monitoring", "scouting", "sampling", "threshold", "cultural control", "cultural practices", "biological control", "natural enemies", "physical control", "mechanical control", "chemical control", "pesticides", "research priorities", "regulatory priorities", "education priorities", "pest management", "weed management", "insect management", "disease management", "management strategies", "worker protection", "pollinator"]
    for sec_name, pats in COMPILED_SECTION_PATTERNS.items():
        match_positions = []
        for pat in pats:
            for m in pat.finditer(lower):
                match_positions.append((m.start(), m.group(0)))
        if match_positions:
            start = min(pos for pos, _ in match_positions)
            next_positions = []
            for candidate in heading_candidates:
                for m in re.finditer(re.escape(candidate), lower):
                    if m.start() > start + 20:
                        next_positions.append(m.start())
            end = min(next_positions) if next_positions else len(joined)
            sections[sec_name] = joined[start:end]
    return sections

def _count_lexicon_hits(text, compiled_patterns):
    if not text:
        return 0
    return sum(len(p.findall(text)) for p in compiled_patterns)

def _section_presence_score(sections):
    core = ["monitoring", "cultural_controls", "biological_controls", "physical_controls", "chemical_controls"]
    present = sum(1 for s in core if s in sections and len((sections.get(s) or "").strip()) > 50)
    return present / len(core)

def _score_document_text(full_text, sections, doc_type):
    full_text = full_text or ""
    sections = sections or {}
    token_denom = max(200, len(re.findall(r"\w+", full_text)))
    sec_cov = _section_presence_score(sections)
    mon_ct = _count_lexicon_hits(full_text, COMPILED_LEXICONS["monitoring"])
    non_ct = _count_lexicon_hits(full_text, COMPILED_LEXICONS["nonchemical"])
    chem_ct = _count_lexicon_hits(full_text, COMPILED_LEXICONS["chemical"])
    dec_ct = _count_lexicon_hits(full_text, COMPILED_LEXICONS["decision_support"])
    dep_ct = _count_lexicon_hits(full_text, COMPILED_LEXICONS["dependency"])
    res_ct = _count_lexicon_hits(full_text, COMPILED_LEXICONS["resistance_management"])
    mon_sec = _count_lexicon_hits(sections.get("monitoring", ""), COMPILED_LEXICONS["monitoring"])
    non_sec = _count_lexicon_hits(sections.get("cultural_controls", ""), COMPILED_LEXICONS["nonchemical"]) + _count_lexicon_hits(sections.get("biological_controls", ""), COMPILED_LEXICONS["nonchemical"]) + _count_lexicon_hits(sections.get("physical_controls", ""), COMPILED_LEXICONS["nonchemical"])
    chem_sec = _count_lexicon_hits(sections.get("chemical_controls", ""), COMPILED_LEXICONS["chemical"])
    pri_sec = _count_lexicon_hits(sections.get("research_priorities", ""), COMPILED_LEXICONS["dependency"])
    mon_rate = _normalize_rate(mon_ct + 2 * mon_sec, token_denom / 1000)
    non_rate = _normalize_rate(non_ct + 2 * non_sec, token_denom / 1000)
    chem_rate = _normalize_rate(chem_ct + 2 * chem_sec, token_denom / 1000)
    dec_rate = _normalize_rate(dec_ct, token_denom / 1000)
    dep_rate = _normalize_rate(dep_ct + 2 * pri_sec, token_denom / 1000)
    res_rate = _normalize_rate(res_ct, token_denom / 1000)
    monitoring_score = _bounded(mon_rate / 4.0)
    nonchemical_score = _bounded(non_rate / 5.0)
    chemical_score = _bounded(chem_rate / 6.0)
    decision_support_score = _bounded(dec_rate / 3.0)
    dependency_score = _bounded(dep_rate / 2.5)
    resistance_management_score = _bounded(res_rate / 2.0)
    ipm_breadth_index = _bounded(0.30 * monitoring_score + 0.35 * nonchemical_score + 0.20 * decision_support_score + 0.15 * resistance_management_score + 0.10 * sec_cov)
    chemical_reliance_index = _bounded(0.60 * chemical_score + 0.25 * dependency_score + 0.15 * max(0.0, chemical_score - nonchemical_score))
    if "pmsp" in str(doc_type or "").lower():
        chemical_reliance_index = _bounded(chemical_reliance_index * 0.95)
    return {"monitoring_score": monitoring_score, "nonchemical_score": nonchemical_score, "chemical_score": chemical_score, "decision_support_score": decision_support_score, "dependency_score": dependency_score, "resistance_management_score": resistance_management_score, "section_coverage_score": sec_cov, "ipm_breadth_index": ipm_breadth_index, "chemical_reliance_index": chemical_reliance_index}

def _combine_text_scores_with_priors(text_scores, priors, text_quality):
    if text_scores is None:
        return {"monitoring_score": priors["monitoring_prior"], "nonchemical_score": priors["nonchemical_prior"], "chemical_score": priors["chemical_prior"], "decision_support_score": priors["decision_support_prior"], "dependency_score": priors["dependency_prior"], "resistance_management_score": priors["resistance_management_prior"], "section_coverage_score": 0.0, "ipm_breadth_index": _bounded(0.30 * priors["monitoring_prior"] + 0.35 * priors["nonchemical_prior"] + 0.20 * priors["decision_support_prior"] + 0.15 * priors["resistance_management_prior"]), "chemical_reliance_index": _bounded(0.60 * priors["chemical_prior"] + 0.25 * priors["dependency_prior"] + 0.15 * max(0.0, priors["chemical_prior"] - priors["nonchemical_prior"]))}
    text_wt = 0.85 if text_quality == "high" else 0.70 if text_quality == "medium" else 0.50
    prior_wt = 1.0 - text_wt
    out = {}
    for score_name, prior_name in [("monitoring_score", "monitoring_prior"), ("nonchemical_score", "nonchemical_prior"), ("chemical_score", "chemical_prior"), ("decision_support_score", "decision_support_prior"), ("dependency_score", "dependency_prior"), ("resistance_management_score", "resistance_management_prior")]:
        out[score_name] = _bounded(text_wt * text_scores[score_name] + prior_wt * priors[prior_name])
    out["section_coverage_score"] = text_scores["section_coverage_score"]
    out["ipm_breadth_index"] = _bounded(0.30 * out["monitoring_score"] + 0.35 * out["nonchemical_score"] + 0.20 * out["decision_support_score"] + 0.15 * out["resistance_management_score"] + 0.10 * out["section_coverage_score"])
    out["chemical_reliance_index"] = _bounded(0.60 * out["chemical_score"] + 0.25 * out["dependency_score"] + 0.15 * max(0.0, out["chemical_score"] - out["nonchemical_score"]))
    return out

def _build_region_state_lookup():
    states_json = _ipm_get("/state")
    state_rows = _ipm_response_to_rows(states_json)
    states = pd.DataFrame(state_rows)
    states.columns = [str(c).strip() for c in states.columns]
    fips_col = next((c for c in states.columns if str(c).lower() in ("fipscode", "fips_code", "statefips", "state_fips", "fips")), None)
    region_col = next((c for c in states.columns if str(c).lower() in ("regionid", "region_id", "cipmregionid")), None)
    region_name_col = next((c for c in states.columns if str(c).lower() == "region"), None)
    if fips_col is None or region_col is None:
        raise KeyError("State table missing FIPS or region. Columns: %s" % list(states.columns))
    states = states[states[fips_col].notna()].copy()
    states["state_fips"] = states[fips_col].astype(str).str.replace(r"\D", "", regex=True).str.zfill(2)
    states = states[(states["state_fips"].str.len() >= 2) & (states["state_fips"] != "00")].copy()
    states["_regionid"] = states[region_col]
    regionid_to_states = states.groupby("_regionid")["state_fips"].apply(lambda x: sorted(set(x))).to_dict()
    regionname_to_states = states.groupby(region_name_col)["state_fips"].apply(lambda x: sorted(set(x))).to_dict() if region_name_col else {}
    return regionid_to_states, regionname_to_states

def _state_fips_for_region(region_name, regionname_to_states):
    if not region_name:
        return [], "low"
    rn = str(region_name).strip()
    if rn in regionname_to_states:
        return list(regionname_to_states[rn]), "medium"
    prefix = rn.lower()[:4]
    matched = sorted({f for k, vals in regionname_to_states.items() if prefix in str(k).lower() for f in vals})
    return matched, "low"

def build_crop_geo_doc_ipm(use_structured_text=USE_STRUCTURED_TEXT, use_pdf_fallback=USE_PDF_FALLBACK):
    regionid_to_states, regionname_to_states = _build_region_state_lookup()
    sources_cp = _ipm_response_to_rows(_ipm_get("/source", {"sourcetypeid": 3}))
    sources_pmsp = _ipm_response_to_rows(_ipm_get("/source", {"sourcetypeid": 4}))
    raw_sources = sources_cp + sources_pmsp
    rows = []
    for s in raw_sources:
        title = _get(s, "source")
        source_id = _get(s, "sourceid")
        url = _get(s, "url")
        doc_type = _get(s, "sourcetype") or ""
        doc_year = _parse_doc_year(_get(s, "sourcedate"))
        crop = _crop_from_title(title)
        crop_family = _crop_family(crop)
        region_id = _get(s, "cipmregionid", "regionid")
        region_name = _get(s, "region")
        state_fips_list = []
        geo_match_confidence = "low"
        source_geo_type = "unknown"
        if region_id is not None and region_id in regionid_to_states:
            state_fips_list = list(regionid_to_states[region_id])
            geo_match_confidence = "high"
            source_geo_type = "region"
        else:
            state_fips_list, geo_match_confidence = _state_fips_for_region(region_name, regionname_to_states)
            source_geo_type = "region"
        state_fips_list = [sf for sf in state_fips_list if sf != "00"]
        if not state_fips_list:
            state_fips_list = ["ALL"]
            source_geo_type = "national_fallback"
        full_text = None
        sections = None
        text_source = "none"
        text_parse_quality = "none"
        if use_structured_text and source_id is not None:
            structured = _fetch_structured_report_text(source_id)
            if structured and structured.get("full_text"):
                full_text = structured.get("full_text")
                sections = structured.get("sections") or _split_sections_from_text(full_text)
                text_source = "html_structured"
                text_parse_quality = "high" if sections else "medium"
        if full_text is None and use_pdf_fallback and url:
            pdf_text = _fetch_pdf_text(url)
            if pdf_text:
                full_text = pdf_text
                sections = _split_sections_from_text(pdf_text)
                text_source = "pdf"
                text_parse_quality = "medium" if sections else "low"
        if full_text is None and source_id is not None:
            try:
                r = requests.get("https://ipmdata.ipmcenters.org/source_report.cfm?sourceid=" + str(source_id) + "&view=yes", timeout=PDF_TIMEOUT)
                if r.ok and len(r.text) > 500:
                    html_text = re.sub(r"<script[^>]*>.*?</script>", " ", r.text, flags=re.DOTALL | re.I)
                    html_text = re.sub(r"<[^>]+>", " ", html_text)
                    html_text = re.sub(r"\s+", " ", html_text).strip()
                    if len(html_text) >= 200:
                        full_text = html_text
                        sections = _split_sections_from_text(full_text)
                        text_source = "html_report"
                        text_parse_quality = "medium" if sections else "low"
            except Exception:
                pass
        priors = _metadata_priors(doc_type=doc_type, doc_year=doc_year, crop=crop)
        text_scores = _score_document_text(full_text, sections, doc_type) if full_text else None
        scores = _combine_text_scores_with_priors(text_scores, priors, text_parse_quality)
        for sf in state_fips_list:
            rows.append({
                "crop": crop,
                "crop_family": crop_family,
                "state_fips": sf,
                "document_year": doc_year,
                "document_type": doc_type,
                "source_id": source_id,
                "source_title": title,
                "url": url,
                "source_geo_type": source_geo_type,
                "geo_match_confidence": geo_match_confidence,
                "text_source": text_source,
                "text_parse_quality": text_parse_quality,
                "monitoring_score": scores["monitoring_score"],
                "nonchemical_score": scores["nonchemical_score"],
                "chemical_score": scores["chemical_score"],
                "decision_support_score": scores["decision_support_score"],
                "dependency_score": scores["dependency_score"],
                "resistance_management_score": scores["resistance_management_score"],
                "section_coverage_score": scores["section_coverage_score"],
                "ipm_breadth_index": scores["ipm_breadth_index"],
                "chemical_reliance_index": scores["chemical_reliance_index"],
            })
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=["crop", "crop_family", "state_fips", "document_year", "document_type", "source_id", "source_title", "url", "source_geo_type", "geo_match_confidence", "text_source", "text_parse_quality", "monitoring_score", "nonchemical_score", "chemical_score", "decision_support_score", "dependency_score", "resistance_management_score", "section_coverage_score", "ipm_breadth_index", "chemical_reliance_index"])
    out["crop_family"] = out["crop_family"].astype(str)
    out["state_fips"] = out["state_fips"].astype(str).str.zfill(2).where(out["state_fips"] != "ALL", "ALL")
    for col, rescaled_col in [("ipm_breadth_index", "ipm_breadth_index_rescaled"), ("chemical_reliance_index", "chemical_reliance_index_rescaled")]:
        if col in out.columns and out[col].notna().any():
            lo, hi = out[col].min(), out[col].max()
            if hi > lo:
                out[rescaled_col] = (out[col] - lo) / (hi - lo)
            else:
                out[rescaled_col] = 0.5 if pd.notna(lo) else np.nan
    return out

def _quality_to_numeric(q):
    return {"none": 0.0, "low": 0.25, "medium": 0.5, "high": 1.0}.get(str(q).lower(), 0.0)

def _confidence_to_numeric(c):
    return {"low": 1/3, "medium": 2/3, "high": 1.0}.get(str(c).lower(), 0.0)

def aggregate_crop_geo_doc_ipm_to_crop_state(crop_geo_doc_ipm, analysis_years=None):
    """Aggregate by (crop, state_fips) and analysis year using recency-weighted means."""
    if crop_geo_doc_ipm is None or len(crop_geo_doc_ipm) == 0:
        return pd.DataFrame()
    analysis_years = analysis_years or ANALYSIS_YEARS_IPM
    df = crop_geo_doc_ipm.copy()
    df["_text_quality_num"] = df["text_parse_quality"].map(_quality_to_numeric) if "text_parse_quality" in df.columns else 0.0
    df["_geo_conf_num"] = df["geo_match_confidence"].map(_confidence_to_numeric) if "geo_match_confidence" in df.columns else 0.0
    df["document_year"] = pd.to_numeric(df["document_year"], errors="coerce")
    out_list = []
    for (crop, state_fips), g in df.groupby(["crop", "state_fips"]):
        crop_family = g["crop_family"].dropna().iloc[0] if "crop_family" in g.columns and g["crop_family"].notna().any() else _crop_family(crop)
        for ay in analysis_years:
            recency = 1.0 / (1.0 + np.abs(g["document_year"] - ay).fillna(99))
            w = recency / recency.sum() if recency.sum() > 0 else recency * 0
            row = {"crop": crop, "crop_family": crop_family, "state_fips": state_fips, "year": ay, "n_docs": len(g)}
            for col in ["ipm_breadth_index", "chemical_reliance_index", "ipm_breadth_index_rescaled", "chemical_reliance_index_rescaled"]:
                if col in g.columns:
                    row[col] = (w * g[col]).sum()
            row["mean_text_quality"] = (w * g["_text_quality_num"]).sum()
            row["mean_geo_confidence"] = (w * g["_geo_conf_num"]).sum()
            row["weighted_doc_age"] = (w * (ay - g["document_year"])).sum()
            out_list.append(row)
    out = pd.DataFrame(out_list)
    if len(out) > 0:
        out["state_fips"] = out["state_fips"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(2).where(out["state_fips"] != "ALL", "ALL")
        out["crop"] = out["crop"].astype(str)
        out["crop_family"] = out["crop_family"].astype(str)
        out["year"] = out["year"].astype(int)
    return out

def aggregate_ipm_match_table(df, group_cols, tier_name):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    value_cols = ["ipm_breadth_index", "chemical_reliance_index", "ipm_breadth_index_rescaled", "chemical_reliance_index_rescaled", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age"]
    rows = []
    for key, g in df.groupby(group_cols):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        weights = pd.to_numeric(g.get("n_docs", 1), errors="coerce").fillna(1).clip(lower=1)
        for col in value_cols:
            if col in g.columns:
                valid = g[col].notna()
                if valid.any():
                    row[col] = np.average(g.loc[valid, col], weights=weights.loc[valid])
                else:
                    row[col] = np.nan
        row["n_docs"] = int(weights.sum())
        row["ipm_match_tier"] = tier_name
        row["ipm_source_crop"] = g["crop"].dropna().iloc[0] if "crop" in g.columns and g["crop"].notna().any() else np.nan
        row["ipm_source_crop_family"] = g["crop_family"].dropna().iloc[0] if "crop_family" in g.columns and g["crop_family"].notna().any() else np.nan
        row["ipm_source_state_fips"] = g["state_fips"].dropna().iloc[0] if "state_fips" in g.columns and g["state_fips"].notna().any() else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def build_ipm_match_tables(crop_state_ipm):
    if crop_state_ipm is None or len(crop_state_ipm) == 0:
        return {}
    exact_state = crop_state_ipm.copy()
    exact_state["ipm_match_tier"] = "exact_crop_state"
    exact_state["ipm_source_crop"] = exact_state["crop"]
    exact_state["ipm_source_crop_family"] = exact_state["crop_family"]
    exact_state["ipm_source_state_fips"] = exact_state["state_fips"]

    state_only = exact_state[exact_state["state_fips"] != "ALL"].copy()
    national_seed = exact_state.copy()
    national_seed["state_fips"] = "ALL"

    return {
        "exact_crop_state": exact_state,
        "crop_family_state": aggregate_ipm_match_table(state_only, ["crop_family", "state_fips", "year"], "crop_family_state"),
        "exact_crop_national": aggregate_ipm_match_table(national_seed, ["crop", "year"], "exact_crop_national"),
        "crop_family_national": aggregate_ipm_match_table(national_seed, ["crop_family", "year"], "crop_family_national"),
    }

def match_county_crop_year_ipm(ccy, crop_state_ipm):
    fill_cols = [
        "ipm_breadth_index", "chemical_reliance_index", "ipm_breadth_index_rescaled", "chemical_reliance_index_rescaled",
        "mean_text_quality", "mean_geo_confidence", "weighted_doc_age", "n_docs",
        "ipm_match_tier", "ipm_source_crop", "ipm_source_crop_family", "ipm_source_state_fips",
    ]
    base = ccy.copy()
    if len(base) == 0:
        return base
    if "crop_family" not in base.columns:
        base["crop_family"] = base["crop"].map(_crop_family)
    for col in fill_cols:
        base[col] = np.nan
    if crop_state_ipm is None or len(crop_state_ipm) == 0:
        return base
    tables = build_ipm_match_tables(crop_state_ipm)
    match_specs = [
        ("exact_crop_state", ["crop", "state_fips", "year"], ["crop", "state_fips", "year"]),
        ("crop_family_state", ["crop_family", "state_fips", "year"], ["crop_family", "state_fips", "year"]),
        ("exact_crop_national", ["crop", "year"], ["crop", "year"]),
        ("crop_family_national", ["crop_family", "year"], ["crop_family", "year"]),
    ]
    for tier_name, left_on, right_on in match_specs:
        table = tables.get(tier_name)
        if table is None or len(table) == 0:
            continue
        tmp = base[left_on].merge(table, left_on=left_on, right_on=right_on, how="left", suffixes=("", "_match"))
        mask = base["ipm_breadth_index"].isna() & tmp["ipm_breadth_index"].notna()
        if not mask.any():
            continue
        for col in fill_cols:
            if col in tmp.columns:
                base.loc[mask, col] = tmp.loc[mask, col].values
    return base

try:
    crop_geo_doc_ipm = build_crop_geo_doc_ipm()
    print("crop_geo_doc_ipm (section-aware, inline):", crop_geo_doc_ipm.shape)
    if len(crop_geo_doc_ipm) > 0 and "text_source" in crop_geo_doc_ipm.columns:
        print("  Text source:", crop_geo_doc_ipm["text_source"].value_counts().to_dict())
    if len(crop_geo_doc_ipm) > 0 and "crop" in crop_geo_doc_ipm.columns:
        print("  Crop coverage:", crop_geo_doc_ipm["crop"].value_counts().head(12).to_dict())
except Exception as e:
    print("build_crop_geo_doc_ipm failed:", e)
    crop_geo_doc_ipm = pd.DataFrame(columns=["crop", "crop_family", "state_fips", "document_year", "document_type", "source_id", "url", "ipm_breadth_index", "chemical_reliance_index", "monitoring_score", "nonchemical_score"])

crop_state_ipm = aggregate_crop_geo_doc_ipm_to_crop_state(crop_geo_doc_ipm, analysis_years=ANALYSIS_YEARS_IPM)

ccy = county_crop_year_ag.copy()
ccy["state_fips"] = ccy["FIPS"].astype(str).str.zfill(5).str[:2]
ccy["crop"] = ccy["crop"].astype(str)
if "crop_family" not in ccy.columns:
    ccy["crop_family"] = ccy["crop"].map(_crop_family)
if "year" in ccy.columns:
    ccy["year"] = pd.to_numeric(ccy["year"], errors="coerce")
    ccy = ccy[ccy["year"].notna()].copy()
    ccy["year"] = ccy["year"].astype(int)
    ccy = ccy[ccy["year"].isin(ANALYSIS_YEARS_IPM)].copy()
if len(crop_state_ipm) > 0:
    crop_state_ipm = crop_state_ipm.copy()
    crop_state_ipm["crop"] = crop_state_ipm["crop"].astype(str)
    crop_state_ipm["crop_family"] = crop_state_ipm["crop_family"].astype(str)
    crop_state_ipm["state_fips"] = crop_state_ipm["state_fips"].astype(str)
    crop_state_ipm["year"] = pd.to_numeric(crop_state_ipm["year"], errors="coerce").astype(int)
    print("crop_state_ipm (for merge):", crop_state_ipm.shape, "| crops:", sorted(crop_state_ipm["crop"].unique().tolist())[:15], "| years:", sorted(crop_state_ipm["year"].dropna().unique().tolist()))
if len(ccy) > 0 and len(crop_state_ipm) > 0:
    county_crop_year_ipm = match_county_crop_year_ipm(ccy, crop_state_ipm)
    n_match = county_crop_year_ipm["ipm_breadth_index"].notna().sum()
    print("county_crop_year_ipm:", county_crop_year_ipm.shape, "| rows with IPM match:", n_match, "(%.1f%%)" % (100 * n_match / len(county_crop_year_ipm) if len(county_crop_year_ipm) else 0))
    if "ipm_match_tier" in county_crop_year_ipm.columns:
        print("  Match tiers:", county_crop_year_ipm["ipm_match_tier"].fillna("unmatched").value_counts().to_dict())
else:
    county_crop_year_ipm = ccy.copy()
    for col in ["ipm_breadth_index", "chemical_reliance_index", "ipm_breadth_index_rescaled", "chemical_reliance_index_rescaled", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age", "n_docs", "ipm_match_tier", "ipm_source_crop", "ipm_source_crop_family", "ipm_source_state_fips"]:
        county_crop_year_ipm[col] = np.nan
if len(ccy) == 0:
    county_crop_year_ipm = pd.DataFrame()

def collapse_county_year(ccy_ipm: pd.DataFrame) -> pd.DataFrame:
    """Acreage- and value-weighted IPM indices; coverage/confidence/age features."""
    if len(ccy_ipm) == 0:
        return pd.DataFrame()
    out = []
    for (fips, year), g in ccy_ipm.groupby(["FIPS", "year"]):
        tot_acres = g["acres"].sum()
        tot_value = g["crop_value"].sum(min_count=1) if "crop_value" in g.columns else np.nan
        share_acres = (g["acres"] / tot_acres) if tot_acres > 0 else pd.Series(0.0, index=g.index)
        has_value = pd.notna(tot_value) and tot_value > 0 and "crop_value" in g.columns
        share_value = (g["crop_value"] / tot_value).fillna(0) if has_value else pd.Series(0.0, index=g.index)
        has_ipm = g["ipm_breadth_index"].notna() if "ipm_breadth_index" in g.columns else pd.Series(False, index=g.index)
        ipm_doc_coverage_share = (share_acres[has_ipm].sum()) if tot_acres > 0 else 0.0
        share_ipm = share_acres[has_ipm]
        wt_sum = share_ipm.sum() if ipm_doc_coverage_share > 0 else 0.0
        def weighted_sum(col, weights):
            if col not in g.columns:
                return np.nan
            valid = g[col].notna()
            if valid.sum() == 0:
                return np.nan
            return (weights[valid] * g.loc[valid, col]).sum()
        def wavg(col):
            if col not in g.columns or wt_sum == 0:
                return np.nan
            return (share_acres[has_ipm] * g.loc[has_ipm, col].fillna(0)).sum() / wt_sum
        primary_match_tier = np.nan
        if "ipm_match_tier" in g.columns and has_ipm.any():
            tier_weights = share_acres[has_ipm].groupby(g.loc[has_ipm, "ipm_match_tier"].fillna("unmatched")).sum()
            if len(tier_weights) > 0:
                primary_match_tier = tier_weights.sort_values(ascending=False).index[0]
        row = {
            "FIPS": fips,
            "year": year,
            "ipm_breadth_acre": weighted_sum("ipm_breadth_index", share_acres),
            "chemical_reliance_acre": weighted_sum("chemical_reliance_index", share_acres),
            "ipm_breadth_acre_rescaled": weighted_sum("ipm_breadth_index_rescaled", share_acres),
            "chemical_reliance_acre_rescaled": weighted_sum("chemical_reliance_index_rescaled", share_acres),
            "ipm_breadth_value": weighted_sum("ipm_breadth_index", share_value) if has_value else np.nan,
            "chemical_reliance_value": weighted_sum("chemical_reliance_index", share_value) if has_value else np.nan,
            "ipm_doc_coverage_share": ipm_doc_coverage_share,
            "mean_text_quality": wavg("mean_text_quality"),
            "mean_geo_confidence": wavg("mean_geo_confidence"),
            "weighted_doc_age": wavg("weighted_doc_age"),
            "county_crop_concentration": (share_acres ** 2).sum() if tot_acres > 0 else np.nan,
            "specialty_crop_share": (g.loc[g["crop_family"] == "specialty_crop", "acres"].sum() / tot_acres) if tot_acres > 0 else 0.0,
            "total_ag_value": tot_value if pd.notna(tot_value) else np.nan,
            "ipm_primary_match_tier": primary_match_tier,
        }
        out.append(row)
    return pd.DataFrame(out)

county_year_collapsed = collapse_county_year(county_crop_year_ipm)
if len(county_year_collapsed) > 0:
    print("County-year collapsed:", county_year_collapsed.shape)
    print("  Columns:", list(county_year_collapsed.columns))
    has_any_value = county_crop_year_ag["crop_value"].notna().any() if "crop_value" in county_crop_year_ag.columns else False
    if not has_any_value:
        print("  Note: ipm_breadth_value / chemical_reliance_value / total_ag_value are all NaN (crop_value not populated; add NASS for value-weighted IPM).")
    if "ipm_breadth_index" in county_crop_year_ipm.columns:
        n_ccy = len(county_crop_year_ipm)
        n_with_ipm = county_crop_year_ipm["ipm_breadth_index"].notna().sum()
        print("  County-crop-year rows with non-null IPM (acre): %.0f / %.0f (%.1f%%)" % (n_with_ipm, n_ccy, 100 * n_with_ipm / n_ccy if n_ccy else 0))
    if "ipm_breadth_acre" in county_year_collapsed.columns:
        n_cy = len(county_year_collapsed)
        n_cy_with_ipm = county_year_collapsed["ipm_breadth_acre"].notna().sum()
        n_cy_nonzero = (county_year_collapsed["ipm_breadth_acre"].notna() & (county_year_collapsed["ipm_breadth_acre"] > 0)).sum()
        n_cy_with_coverage = (county_year_collapsed["ipm_doc_coverage_share"] > 0).sum() if "ipm_doc_coverage_share" in county_year_collapsed.columns else 0
        print("  County-year rows with non-null ipm_breadth_acre: %d / %d (%.1f%%)" % (n_cy_with_ipm, n_cy, 100 * n_cy_with_ipm / n_cy if n_cy else 0))
        print("  County-year rows with ipm_breadth_acre > 0: %d" % n_cy_nonzero)
        print("  County-year rows with any IPM document coverage: %d / %d (%.1f%%)" % (n_cy_with_coverage, n_cy, 100 * n_cy_with_coverage / n_cy if n_cy else 0))
        if "ipm_primary_match_tier" in county_year_collapsed.columns:
            print("  Primary county-year match tiers:", county_year_collapsed["ipm_primary_match_tier"].fillna("unmatched").value_counts().to_dict())
else:
    if len(county_crop_year_ag) > 0:
        ccy_ag = county_crop_year_ag.copy()
        for c in ["ipm_breadth_index", "chemical_reliance_index"]:
            ccy_ag[c] = np.nan
        county_year_collapsed = collapse_county_year(ccy_ag)
        print("County-year collapsed (no IPM yet):", county_year_collapsed.shape)


crop_geo_doc_ipm (section-aware, inline): (19203, 23)
  Text source: {'html_report': 19203}
  Crop coverage: {'other_crop': 11030, 'fruit_veg': 5036, 'corn': 998, 'cotton': 470, 'wheat': 468, 'hay': 432, 'soybean': 379, 'sorghum': 208, 'rice': 182}
crop_state_ipm (for merge): (834, 12) | crops: ['corn', 'cotton', 'fruit_veg', 'hay', 'other_crop', 'rice', 'sorghum', 'soybean', 'wheat'] | years: [2018, 2019]
county_crop_year_ipm: (143414, 21) | rows with IPM match: 0 (0.0%)
  Match tiers: {'unmatched': 143414}
County-year collapsed: (6074, 16)
  Columns: ['FIPS', 'year', 'ipm_breadth_acre', 'chemical_reliance_acre', 'ipm_breadth_acre_rescaled', 'chemical_reliance_acre_rescaled', 'ipm_breadth_value', 'chemical_reliance_value', 'ipm_doc_coverage_share', 'mean_text_quality', 'mean_geo_confidence', 'weighted_doc_age', 'county_crop_concentration', 'specialty_crop_share', 'total_ag_value', 'ipm_primary_match_tier']
  Note: ipm_breadth_value / chemical_reliance_value / total_ag_value are all Na

## 4. Load CDC PLACES (county-year, 2018 and 2019 only)

Uses **CDCPLACES R package** via `rpy2`. County-level for **2018 and 2019 only** (release 2020 = 2018 BRFSS, 2021 = 2019 BRFSS). One row per county per year.

"Merge to joint on FIPS and YEAR; 2016/2017 rows get NaN for PLACES.\n",

**Measures:** CASTHMA, COPD, CSMOKING, OBESITY, DIABETES.

In [12]:
# CDC PLACES: county-level health outcomes for 2018 and 2019 (release 2020 = 2018 BRFSS, 2021 = 2019 BRFSS)
# Note: 2020 API has no locationid; 2021 has locationid (FIPS). We fetch 2021 first to build (stateabbr, locationname)->FIPS lookup for 2020.
PLACES_MEASURES = ["CASTHMA", "COPD", "CSMOKING", "OBESITY", "DIABETES"]
PLACES_RELEASE_TO_YEAR = {"2021": 2019, "2020": 2018}  # 2021 first so we have FIPS lookup for 2020

try:
    cdcplaces = importr("CDCPLACES")
    r.assign("measures_r", StrVector(PLACES_MEASURES))
    tmp_csv = os.path.join(tempfile.gettempdir(), "places_county_r.csv")
    r.assign("tmp_path", tmp_csv)
    places_list = []
    name_to_fips = None  # (stateabbr, locationname) -> FIPS from 2021 data
    for release_str, analysis_year in PLACES_RELEASE_TO_YEAR.items():
        r.assign("release_r", release_str)
        r('places_county <- get_places(geography = "county", state = NULL, measure = measures_r, release = release_r, geometry = FALSE)')
        r('''
        atomic <- sapply(places_county, is.atomic)
        places_county <- places_county[, atomic, drop = FALSE]
        write.csv(places_county, tmp_path, row.names = FALSE)
        ''')
        county_df = pd.read_csv(tmp_csv)
        county_df.columns = [c.strip().lower().replace(" ", "_") for c in county_df.columns]
        measure_col = "measureid" if "measureid" in county_df.columns else [c for c in county_df.columns if "measure" in c][0]
        value_col = "data_value" if "data_value" in county_df.columns else [c for c in county_df.columns if "value" in c and "conf" not in c][0]
        id_col = next((c for c in ["locationid", "location_id", "geolocation", "countyfips", "fips"] if c in county_df.columns), None)
        has_locationid = id_col is not None
        if not has_locationid:
            id_col = [c for c in county_df.columns if "location" in c or "fips" in c or c == "geoid"][0]
        if has_locationid:
            county_wide = county_df.pivot_table(index=id_col, columns=measure_col, values=value_col, aggfunc="first").reset_index()
            county_wide["FIPS"] = county_wide[id_col].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)
            if name_to_fips is None and "stateabbr" in county_df.columns and "locationname" in county_df.columns:
                name_to_fips = county_df[["stateabbr", "locationname", id_col]].drop_duplicates()
                name_to_fips["FIPS"] = name_to_fips[id_col].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)
                name_to_fips = name_to_fips[["stateabbr", "locationname", "FIPS"]].drop_duplicates()
        else:
            if "stateabbr" not in county_df.columns or id_col not in county_df.columns:
                continue
            county_wide = county_df.pivot_table(index=["stateabbr", id_col], columns=measure_col, values=value_col, aggfunc="first").reset_index()
            county_wide = county_wide.rename(columns={id_col: "locationname"})
            if name_to_fips is not None:
                county_wide = county_wide.merge(name_to_fips, on=["stateabbr", "locationname"], how="left")
                county_wide = county_wide.dropna(subset=["FIPS"])
                county_wide = county_wide.drop(columns=["stateabbr", "locationname"], errors="ignore")
            else:
                county_wide["FIPS"] = ""
        keep_cols = ["FIPS"] + [c for c in PLACES_MEASURES if c in county_wide.columns]
        df = county_wide[keep_cols].copy()
        df["YEAR"] = analysis_year
        places_list.append(df)
    try:
        os.remove(tmp_csv)
    except OSError:
        pass
    places = pd.concat(places_list, ignore_index=True)
    print("Loaded CDC PLACES (county-year): 2018 and 2019 only (releases 2020, 2021).")
except Exception as e:
    print(f"CDCPLACES failed: {e}")
    places = pd.DataFrame(columns=["FIPS", "YEAR", "CASTHMA", "COPD", "CSMOKING", "OBESITY", "DIABETES"])

# Normalize so merge on FIPS+YEAR matches: FIPS as 5-char string, YEAR as int
if len(places) > 0 and "FIPS" in places.columns and "YEAR" in places.columns:
    places = places.copy()
    places["FIPS"] = places["FIPS"].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)
    places = places[places["FIPS"].str.len() == 5].copy()  # drop invalid FIPS
    places["YEAR"] = pd.to_numeric(places["YEAR"], errors="coerce")
    places = places[places["YEAR"].notna()].copy()
    places["YEAR"] = places["YEAR"].astype(int)
    places = places.drop_duplicates(subset=["FIPS", "YEAR"])
    print("PLACES years present:", sorted(places["YEAR"].unique().tolist()))

print("PLACES (county-year):", places.shape)
places.head()

CDCPLACES failed: name 'importr' is not defined
PLACES (county-year): (0, 7)


,FIPS,YEAR,CASTHMA,COPD,CSMOKING,OBESITY,DIABETES


In [13]:
places.tail(1000)

,FIPS,YEAR,CASTHMA,COPD,CSMOKING,OBESITY,DIABETES


## 5. Join All Datasets (County–Year on FIPS)

In [14]:
# Focus on 2018 and 2019 only (years with CDC PLACES data)
ANALYSIS_YEARS_JOINT = [2018, 2019]
joint = pest_county[pest_county["YEAR"].isin(ANALYSIS_YEARS_JOINT)].copy()
joint["YEAR"] = joint["YEAR"].astype(int)
joint["FIPS"] = joint["FIPS"].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)

# Merge census and cropland on FIPS (county-level; repeated for each year); merge PLACES on FIPS + YEAR
census_cols = ["FIPS", "NAME", "population", "median_age", "median_income"]
census_cols += [c for c in ["pct_white", "pct_black", "pct_asian", "pct_hispanic", "nchs_urban_rural"] if c in census.columns]
census_sub = census[census_cols].copy()
joint = joint.merge(census_sub, on="FIPS", how="left", suffixes=("", "_census"))

# Merge cropland (left join)
joint = joint.merge(cropland, on="FIPS", how="left")

# Merge PLACES (on FIPS and YEAR; both must be same type for match)
if "YEAR" in places.columns and len(places) > 0:
    joint = joint.merge(places, on=["FIPS", "YEAR"], how="left")
    if "CASTHMA" in joint.columns:
        print("PLACES merge by year - rows with non-null CASTHMA:", joint.groupby("YEAR")["CASTHMA"].apply(lambda x: x.notna().sum()).to_dict())
else:
    joint = joint.merge(places, on="FIPS", how="left") if len(places) > 0 else joint
joint["state_fips"] = joint["FIPS"].astype(str).str.zfill(5).str[:2]

# Merge county-year collapsed crop/IPM indices (acreage- and value-weighted; primary crop representation for IPM)
if len(county_year_collapsed) > 0:
    cyc = county_year_collapsed.rename(columns={"year": "YEAR"})
    joint = joint.merge(cyc, on=["FIPS", "YEAR"], how="left")
    print("Merged county_year_collapsed (ipm_breadth_acre, chemical_reliance_acre, county_crop_concentration, specialty_crop_share, etc.)")

print("Joint dataset shape (county-year rows):", joint.shape)
print("  Years:", sorted(joint["YEAR"].dropna().unique().tolist()))
print("\nColumn coverage:")
print(joint.notna().sum())
joint.head(10)

Merged county_year_collapsed (ipm_breadth_acre, chemical_reliance_acre, county_crop_concentration, specialty_crop_share, etc.)
Joint dataset shape (county-year rows): (6128, 483)
  Years: [2018, 2019]

Column coverage:
FIPS                         6128
YEAR                         6128
pesticide_total_kg           6128
pesticide_anilide_kg         6128
pesticide_carbamate_kg       6128
                             ... 
weighted_doc_age                0
county_crop_concentration    6070
specialty_crop_share         6070
total_ag_value                  0
ipm_primary_match_tier          0
Length: 483, dtype: int64


,FIPS,YEAR,pesticide_total_kg,pesticide_anilide_kg,pesticide_carbamate_kg,pesticide_chlorophenoxy_kg,pesticide_organochlorine_kg,pesticide_organophosphate_kg,pesticide_other_kg,pesticide_pyrethroid_kg,...,ipm_breadth_value,chemical_reliance_value,ipm_doc_coverage_share,mean_text_quality,mean_geo_confidence,weighted_doc_age,county_crop_concentration,specialty_crop_share,total_ag_value,ipm_primary_match_tier
0,01001,2018,34773.85,3690.60,397.30,6995.35,0.0,2033.75,20854.80,119.65,...,NaN,NaN,0.0,NaN,NaN,NaN,0.520671,0.000527,NaN,NaN
1,01001,2019,21120.00,2582.55,9.70,5230.55,0.0,763.95,11639.25,16.80,...,NaN,NaN,0.0,NaN,NaN,NaN,0.520671,0.000527,NaN,NaN
2,01003,2018,277331.10,61192.90,1327.20,10561.50,0.0,4287.00,185663.25,1982.75,...,NaN,NaN,0.0,NaN,NaN,NaN,0.154430,0.008605,NaN,NaN
3,01003,2019,125643.65,30532.30,24.45,4197.00,0.0,7219.45,75275.55,127.30,...,NaN,NaN,0.0,NaN,NaN,NaN,0.154430,0.008605,NaN,NaN
4,01005,2018,63256.35,3484.60,306.80,11760.30,0.0,1455.95,42686.50,626.70,...,NaN,NaN,0.0,NaN,NaN,NaN,0.324831,0.000003,NaN,NaN
5,01005,2019,28806.30,3311.45,13.90,7829.15,0.0,1429.95,15947.15,18.60,...,NaN,NaN,0.0,NaN,NaN,NaN,0.324831,0.000003,NaN,NaN
6,01007,2018,11299.05,1073.55,0.00,5137.15,0.0,249.05,3777.65,53.65,...,NaN,NaN,0.0,NaN,NaN,NaN,0.578789,0.000042,NaN,NaN
7,01007,2019,7188.60,1475.40,0.30,1985.35,0.0,11.60,3437.35,4.00,...,NaN,NaN,0.0,NaN,NaN,NaN,0.578789,0.000042,NaN,NaN
8,01009,2018,33089.65,3647.45,53.40,11166.35,0.0,307.40,17472.10,77.55,...,NaN,NaN,0.0,NaN,NaN,NaN,0.493315,0.000333,NaN,NaN
9,01009,2019,29893.75,4183.00,25.75,18218.00,0.0,378.20,6518.10,1.80,...,NaN,NaN,0.0,NaN,NaN,NaN,0.493315,0.000333,NaN,NaN


## 6. Save joint dataset

Write the joined table to CSV for modeling and for **`joint_eda.ipynb`**.


In [15]:
out_path = "../data/joint_county_year_2018_2019.csv"
joint.to_csv(out_path, index=False)
print("Saved:", out_path, "(", len(joint), "rows )")


Saved: ../data/joint_county_year_2018_2019.csv ( 6128 rows )
